In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from functools import reduce

In [5]:
df = pd.read_csv("../data/dataset_mood_smartphone.csv", index_col=0)
df["time"] = pd.to_datetime(df["time"])

In [6]:
#kunnen we later ook nog makkelijk aanpassen naar max ofzo 
# Bijvoorbeeld voor "activity", "circumplex.arousal", "circumplex.valence"
def get_agg(var):
    if var in ["mood", "activity", "circumplex.arousal", "circumplex.valence"]:
        return "mean"
    
    elif "appCat" in var:
        return "sum"
    
    elif var in ["screen","sms", "call"]:
        return "sum"
    
    else:
        return "mean"  # fallback

In [22]:
# hoeveel uur/dagen per keer
freq = "D"  #of "h", "3h", "6h", "D" etc.

user_datasets = {}

all_variables = df["variable"].unique()

for user_id in df["id"].unique():
    
    df_user = df[df["id"] == user_id].copy()
    df_user["time"] = pd.to_datetime(df_user["time"])
    
    # -----------------------------
    # mood-based range
    # -----------------------------
    df_mood = df_user[df_user["variable"] == "mood"]
    
    if df_mood.empty:
        print(f"{user_id}: geen mood data -> skip")
        continue
    
    start = df_mood["time"].min().floor("D")
    end = df_mood["time"].max().ceil("D")
    
    full_index = pd.date_range(start=start, end=end, freq=freq)
    
    # -----------------------------
    # pivot
    # -----------------------------
    df_pivot = df_user.pivot_table(
        index="time",
        columns="variable",
        values="value",
        aggfunc="mean"
    ).sort_index()

    df_pivot = df_pivot.reindex(columns=all_variables)
    
    # -----------------------------
    # resample per variabele
    # -----------------------------
    df_resampled = pd.DataFrame(index=full_index)

    for col in df_pivot.columns:
        agg = get_agg(col)
        
        if agg == "mean":
            df_resampled[col] = df_pivot[col].resample(freq).mean()
        
        elif agg == "sum":
            df_resampled[col] = df_pivot[col].resample(freq).sum().fillna(0) #geen waarde, dus geen occurence dus 0


    # index -> kolom
    df_resampled = df_resampled.reset_index().rename(columns={"index": "time_bin"})
    
    # -----------------------------
    # opslaan per user
    # -----------------------------
    user_datasets[user_id] = df_resampled

Het is dus een dictonary met user ID als namen en het df als item, voorbeeld:

In [24]:
user_datasets['AS14.02'].head()

,time_bin,mood,circumplex.arousal,circumplex.valence,activity,screen,call,sms,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,appCat.social,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather
0,2014-03-16,6.333333,0.0,0.00,NaN,4902.627000,4.0,1.0,2437.046,3151.829,88.392,0.0,0.0,0.0,408.106,181.591,0.000,0.0,0.0,0.0
1,2014-03-17,6.750000,0.5,0.50,0.168068,10159.769001,4.0,0.0,1229.347,4518.116,2511.178,0.0,0.0,0.0,103.699,103.028,0.000,0.0,0.0,0.0
2,2014-03-18,8.200000,0.6,0.80,0.219484,4512.471001,15.0,1.0,2604.521,3933.081,88.943,0.0,0.0,0.0,88.722,212.364,0.000,0.0,0.0,0.0
3,2014-03-19,6.800000,0.4,0.40,0.315632,8182.757000,5.0,1.0,1554.558,2596.989,196.139,0.0,0.0,0.0,105.536,47.815,567.165,0.0,0.0,0.0
4,2014-03-20,7.250000,-0.5,0.75,0.239025,4360.334001,10.0,1.0,1777.977,2154.371,148.180,0.0,0.0,0.0,86.857,131.598,0.000,0.0,0.0,0.0
